In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
import logging
import pickle

import mlflow
import pandas as pd

from mlflow import MlflowClient

from databricks.feature_engineering import (
    FeatureEngineeringClient,
    FeatureLookup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [0]:
logger = logging.getLogger("DemandPrediction")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )

    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False

In [0]:
class ChampionChallenger:

    def __init__(
        self,
        spark,
        feature_client,
        feature_table,
        output_table,
        model_name
    ):

        self.spark = spark
        self.feature_client = feature_client
        self.feature_table = feature_table
        self.output_table = output_table
        self.model_name = model_name

        mlflow.set_registry_uri(
            "databricks-uc"
        )

        self.client = MlflowClient()

        logger.info(
            "ChampionChallenger initialized"
        )

    def get_test_data(self):

        logger.info(
            "Creating test dataset"
        )

        training_set = self.feature_client.create_training_set(
            df=self.spark.table(self.output_table),
            feature_lookups=[
                FeatureLookup(
                    table_name=self.feature_table,
                    lookup_key="primary_key"
                )
            ],
            label="total_demand",
            exclude_columns=["primary_key"]
        )

        df = training_set.load_df()

        df = df.orderBy(
            "transaction_date"
        )

        df = df.drop(
            "transaction_date"
        )

        df = df.toPandas()

        train_validate_df,test_df = train_test_split(
            df,
            test_size=0.20,
            shuffle=False
        )

        train_df,validate_df = train_test_split(
            train_validate_df,
            test_size=0.20,
            shuffle=False
        )

        X_test = test_df.drop(
            columns=["total_demand"]
        )

        y_test = test_df["total_demand"]

        return X_test,y_test

    def load_encoder(self,run_id):

        logger.info(
            "Loading encoder"
        )

        encoder_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path="preprocessing/onehot_encoder.pkl"
        )

        with open(
            encoder_path,
            "rb"
        ) as file:

            encoder = pickle.load(file)

        return encoder
    
    def encode_test_data(self,X_test,encoder):

        category_columns = [
            "product_name",
            "destination_city"
        ]

        numeric_columns = [
            "avg_unit_price",
            "total_inventory",
            "transaction_count",
            "demand_1",
            "demand_7",
            "rolling_7_day_avg",
            "month",
            "day_of_week"
        ]

        encoded_data = encoder.transform(
            X_test[category_columns]
        )

        encoded_columns = (
            encoder.get_feature_names_out(
                category_columns
            )
        )

        encoded_df = pd.DataFrame(
            encoded_data,
            columns=encoded_columns,
            index=X_test.index
        )

        X_test_final = pd.concat(
            [
                X_test[numeric_columns],
                encoded_df
            ],
            axis=1
        )

        return X_test_final
    
    def evaluate_model(self,model_alias,X_test,y_test):

        logger.info(
            "Evaluating model using alias: %s",
            model_alias
        )

        model_details = (
            self.client.get_model_version_by_alias(
                self.model_name,
                model_alias
            )
        )

        model_uri = (
            f"models:/{self.model_name}@{model_alias}"
        )

        model = mlflow.sklearn.load_model(
            model_uri
        )

        encoder = self.load_encoder(
            model_details.run_id
        )

        X_test_final = self.encode_test_data(
            X_test,
            encoder
        )

        predictions = model.predict(
            X_test_final
        )

        test_r2 = r2_score(
            y_test,
            predictions
        )

        logger.info(
            "%s Test R²: %.4f",
            model_alias,
            test_r2
        )

        return {
            "version": model_details.version,
            "run_id": model_details.run_id,
            "test_r2": test_r2,
            "predictions": predictions
        }
    
    def compare_models(self,X_test,y_test):

        logger.info("Starting Champion-Challenger comparison")

        challenger = self.evaluate_model(
            "Challenger",
            X_test,
            y_test
        )

        challenger_r2 = challenger["test_r2"]

        try:

            champion_model = (
                self.client.get_model_version_by_alias(
                    self.model_name,
                    "Champion"
                )
            )

            logger.info(
                "Champion model exists. Version: %s",
                champion_model.version
            )

            champion = self.evaluate_model(
                "Champion",
                X_test,
                y_test
            )

            champion_r2 = champion["test_r2"]

            logger.info(
                "Champion R²: %.4f | Challenger R²: %.4f",
                champion_r2,
                challenger_r2
            )

            metric_r2_passed = (
                challenger_r2 > champion_r2
            )

        except Exception:

            logger.info(
                "No Champion found. "
                "Accepting Challenger as first Champion."
            )

            champion_r2 = None

            metric_r2_passed = True

        self.client.set_model_version_tag(
            name=self.model_name,
            version=challenger["version"],
            key="test_r2",
            value=str(challenger_r2)
        )

        self.client.set_model_version_tag(
            name=self.model_name,
            version=challenger["version"],
            key="metric_r2_passed",
            value=str(metric_r2_passed)
        )

        if champion_r2 is not None:

            self.client.set_model_version_tag(
                name=self.model_name,
                version=challenger["version"],
                key="champion_r2",
                value=str(champion_r2)
            )

        if metric_r2_passed:

            self.client.set_registered_model_alias(
                name=self.model_name,
                alias="Champion",
                version=challenger["version"]
            )

            logger.info(
                "Challenger version %s promoted to Champion",
                challenger["version"]
            )

        else:

            logger.info(
                "No model promotion. "
                "Champion remains unchanged."
            )

        return {
            "challenger_version": challenger["version"],
            "challenger_r2": challenger_r2,
            "champion_r2": champion_r2,
            "metric_r2_passed": metric_r2_passed
        }

    def run(self):

        logger.info(
            "Champion-Challenger pipeline started"
        )

        X_test,y_test = self.get_test_data()

        result = self.compare_models(
            X_test,
            y_test
        )

        logger.info(
            "Champion-Challenger pipeline completed"
        )

        return result


In [0]:
fe = FeatureEngineeringClient()
obj = ChampionChallenger(
    spark=spark,
    feature_client=fe,
    feature_table="oag.gold.demand_input_features",
    output_table="oag.gold.demand_output",
    model_name="oag.ml.demand_prediction_model"
)
result = obj.run()
print(result)